In [42]:
'''
The following updates are to do done in this version
- Divide into code bloack
- Add Residuals vs fitted lines plot for oiginal and transformed variables
- Use PCR with cumulative threshold with 85% variance explained
'''

import os

# Create the output directory if it doesn't exist
output_dir = '/mnt/user-data/outputs'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")
else:
    print(f"Directory already exists: {output_dir}")

Directory already exists: /mnt/user-data/outputs


In [43]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.metrics import (mean_squared_error, r2_score, mean_absolute_error,
                            mean_absolute_percentage_error)
from scipy import stats
from scipy.stats import f_oneway, ttest_ind
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import datetime

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION & STYLING
# ═══════════════════════════════════════════════════════════════════════════════

DARK_BG = "#0d1117"
PANEL_BG = "#161b22"
ACCENT1 = "#58a6ff"
ACCENT2 = "#f78166"
ACCENT3 = "#3fb950"
ACCENT4 = "#d2a8ff"
ACCENT5 = "#ffa657"
GRID_CLR = "#30363d"
TEXT_CLR = "#e6edf3"
SUBTLE = "#8b949e"

def apply_theme():
    plt.rcParams.update({
        "figure.facecolor": DARK_BG,
        "axes.facecolor": PANEL_BG,
        "axes.edgecolor": GRID_CLR,
        "axes.labelcolor": TEXT_CLR,
        "axes.titlecolor": TEXT_CLR,
        "axes.grid": True,
        "grid.color": GRID_CLR,
        "grid.linewidth": 0.5,
        "xtick.color": SUBTLE,
        "ytick.color": SUBTLE,
        "text.color": TEXT_CLR,
        "legend.facecolor": PANEL_BG,
        "legend.edgecolor": GRID_CLR,
        "font.family": "monospace",
        "font.size": 10,
    })

def print_section(title):
    bar = "═" * 100
    print(f"\n{bar}\n  {title}\n{bar}")

def print_subsection(title):
    print(f"\n{'─' * 100}\n  → {title}\n{'─' * 100}")

def calculate_vif(X_data):
    """Calculate Variance Inflation Factor for multicollinearity check."""
    vif_data = []

    for i in range(X_data.shape[1]):
        # Get all features except i-th
        X_temp = X_data.drop(X_data.columns[i], axis=1)
        y_temp = X_data[X_data.columns[i]]

        # Fit linear regression
        lr = LinearRegression()
        lr.fit(X_temp, y_temp)
        r2 = r2_score(y_temp, lr.predict(X_temp))

        # Calculate VIF
        if r2 < 0.9999:
            vif = 1 / (1 - r2)
        else:
            vif = np.inf

        vif_data.append({
            'Feature': X_data.columns[i],
            'VIF': vif
        })

    vif_df = pd.DataFrame(vif_data)
    return vif_df.sort_values('VIF', ascending=False)

def test_ols_significance(X_data, y_data, model_name="OLS"):
    """Perform F-test and T-tests for OLS model variable significance."""
    # Add constant for statsmodels
    import statsmodels.api as sm

    X_with_const = sm.add_constant(X_data)
    ols_model = sm.OLS(y_data, X_with_const).fit()

    return ols_model

In [44]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: DATA LOADING & PREPROCESSING
# ═══════════════════════════════════════════════════════════════════════════════

def load_and_prepare_data():
    """Load CSV and perform initial data cleaning."""
    print_section("STEP 1: DATA LOADING & PREPARATION")

    df = pd.read_csv("/content/__ecommerce_customer_churn_dataset.csv")

    print(f"\n  Initial Dataset:")
    print(f"    • Shape: {df.shape}")
    print(f"    • Columns: {df.shape[1]}")
    print(f"    • Missing values: {df.isnull().sum().sum()}")

    # ── Handle Missing Values ──
    print(f"\n  Missing value handling:")

    # First pass: impute numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        missing_count = df[col].isnull().sum()
        if missing_count > 0:
            df[col].fillna(df[col].median(), inplace=True)
            print(f"    • {col}: {missing_count} values → median imputation")

    # Second pass: impute categorical columns
    categorical_cols = df.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        missing_count = df[col].isnull().sum()
        if missing_count > 0:
            if df[col].nunique() <= 50:  # Only impute if not too many categories
                mode_val = df[col].mode()
                if len(mode_val) > 0:
                    df[col].fillna(mode_val[0], inplace=True)
                    print(f"    • {col}: {missing_count} values → mode imputation")

    # Final check for any remaining NaN
    remaining_nan = df.isnull().sum().sum()
    print(f"  ✓ Remaining NaN values: {remaining_nan}")

    return df

In [45]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════════════════════════

def feature_engineering(df):
    """Create engineered features."""
    print_section("STEP 2: FEATURE ENGINEERING")

    print(f"\n  Creating new features:")

    # Engagement Index
    df['Engagement_Index'] = df['Session_Duration_Avg'] * df['Pages_Per_Session']
    print(f"    ✓ Engagement_Index = Session_Duration_Avg × Pages_Per_Session")

    # Purchase Recency
    df['Purchase_Recency'] = 1 / (df['Days_Since_Last_Purchase'] + 1)
    print(f"    ✓ Purchase_Recency = 1 / (Days_Since_Last_Purchase + 1)")

    # Loyalty Ratio
    df['Loyalty_Ratio'] = df['Total_Purchases'] / (df['Membership_Years'] + 1)
    print(f"    ✓ Loyalty_Ratio = Total_Purchases / (Membership_Years + 1)")

    print(f"\n  New feature statistics:")
    new_features = ['Engagement_Index', 'Purchase_Recency', 'Loyalty_Ratio']
    print(df[new_features].describe().to_string())

    return df

In [46]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: DESCRIPTIVE STATISTICS
# ═══════════════════════════════════════════════════════════════════════════════

def descriptive_statistics(df):
    """Generate comprehensive descriptive statistics."""
    print_section("STEP 3: DESCRIPTIVE STATISTICS & EXPLORATORY ANALYSIS")

    target = 'Lifetime_Value'

    print(f"\n  Target Variable (Lifetime_Value) Statistics:")
    print(f"    • Mean: ${df[target].mean():.2f}")
    print(f"    • Median: ${df[target].median():.2f}")
    print(f"    • Std Dev: ${df[target].std():.2f}")
    print(f"    • Min: ${df[target].min():.2f}")
    print(f"    • Max: ${df[target].max():.2f}")
    print(f"    • Skewness: {stats.skew(df[target]):.4f}")
    print(f"    • Kurtosis: {stats.kurtosis(df[target]):.4f}")

    # Numeric features statistics
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"\n  Numeric Features Summary:")
    print(f"    • Total numeric features: {len(numeric_cols)}")
    print(f"    • Features with high skewness (|skew| > 1): {sum(abs(stats.skew(df[col])) > 1 for col in numeric_cols)}")

    return numeric_cols

In [47]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: PREPARE DATA FOR MODELING
# ═══════════════════════════════════════════════════════════════════════════════

def prepare_model_data(df):
    """Prepare features and target for modeling."""
    print_section("STEP 4: MODEL DATA PREPARATION")

    # Drop non-numeric columns that won't be used
    df_model = df.drop(['City', 'Country'], axis=1, errors='ignore')

    # Handle any NaN in engineered features
    numeric_cols = df_model.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df_model[col].isnull().any():
            df_model[col].fillna(df_model[col].median(), inplace=True)

    # Encode categorical variables
    df_encoded = pd.get_dummies(df_model, columns=['Gender', 'Signup_Quarter'], drop_first=True)

    # Define target and predictors
    target = 'Lifetime_Value'
    target_log = np.log1p(df_encoded[target])

    predictors = [col for col in df_encoded.columns
                  if col not in [target, 'Churned']]

    X = df_encoded[predictors].copy()
    y = df_encoded[target].copy()

    # Final NaN check and removal
    print(f"\n  Final data cleaning:")
    initial_rows = len(X)

    # Remove any rows with NaN
    mask = X.isnull().any(axis=1) | y.isnull()
    X = X[~mask]
    y = y[~mask]
    y_log = np.log1p(y)

    rows_removed = initial_rows - len(X)
    if rows_removed > 0:
        print(f"    • Removed {rows_removed} rows with NaN values")

    # Ensure no NaN in X or y
    assert not X.isnull().any().any(), "X still contains NaN values"
    assert not y.isnull().any(), "y still contains NaN values"

    print(f"  ✓ No NaN values in final dataset")
    print(f"\n  Data shapes:")
    print(f"    • Features (X): {X.shape}")
    print(f"    • Target (y): {y.shape}")
    print(f"    • Number of predictors: {len(predictors)}")

    return X, y, y_log, predictors

In [48]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: OLS ON ORIGINAL VARIABLES
# ═══════════════════════════════════════════════════════════════════════════════

def run_ols_original(X, y, predictors):
    """Run OLS on original variables."""
    print_section("STEP 5: OLS REGRESSION (ORIGINAL VARIABLES)")

    X_orig = X.copy()

    # Fit OLS
    ols_model = LinearRegression()
    ols_model.fit(X_orig, y)

    y_pred = ols_model.predict(X_orig)
    residuals = y - y_pred

    # Calculate metrics
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    mae = mean_absolute_error(y, y_pred)
    mape = mean_absolute_percentage_error(y, y_pred)
    adj_r2 = 1 - (1 - r2) * (len(y) - 1) / (len(y) - X.shape[1] - 1)

    print(f"\n  Model Performance:")
    print(f"    • R² Score: {r2:.6f}")
    print(f"    • Adjusted R²: {adj_r2:.6f}")
    print(f"    • RMSE: ${rmse:.2f}")
    print(f"    • MAE: ${mae:.2f}")
    print(f"    • MAPE: {mape:.6f}")

    print(f"\n  Residual Analysis:")
    print(f"    • Mean: {residuals.mean():.6f}")
    print(f"    • Std Dev: {residuals.std():.6f}")
    print(f"    • Skewness: {stats.skew(residuals):.6f}")
    print(f"    • Kurtosis: {stats.kurtosis(residuals):.6f}")

    # 10-fold CV
    cv_scores = cross_val_score(ols_model, X_orig, y, cv=KFold(n_splits=10, shuffle=True, random_state=42), scoring='r2')
    print(f"\n  10-Fold Cross-Validation:")
    print(f"    • Mean R²: {cv_scores.mean():.6f}")
    print(f"    • Std Dev: {cv_scores.std():.6f}")
    print(f"    • Fold scores: {[f'{s:.4f}' for s in cv_scores]}")

    # Statistical Significance Testing (manual implementation)
    print_subsection("Statistical Significance Testing (OLS - Original Variables)")

    # Add constant for significance testing
    X_numeric = X_orig.values.astype(float)
    X_with_const = np.column_stack([np.ones(len(X_numeric)), X_numeric])

    # Calculate coefficients
    beta = np.linalg.lstsq(X_with_const, y, rcond=None)[0]
    y_pred_sig = X_with_const @ beta
    residuals_sig = y - y_pred_sig

    # Calculate residual sum of squares and standard error
    rss = np.sum(residuals_sig ** 2)
    n = len(y)
    k = X_with_const.shape[1]
    dof = n - k
    mse = rss / dof

    # Calculate standard errors
    try:
        var_covar = mse * np.linalg.inv(X_with_const.T @ X_with_const)
        std_errors = np.sqrt(np.diag(var_covar))
    except:
        std_errors = np.ones(k) * np.sqrt(mse)

    # Calculate t-statistics and p-values
    t_stats = beta / (std_errors + 1e-10)
    p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), dof))

    # Calculate F-statistic
    ss_total = np.sum((y - y.mean()) ** 2)
    ss_res = rss
    ss_reg = ss_total - ss_res
    f_stat = (ss_reg / (k - 1)) / (ss_res / dof)
    f_pvalue = 1 - stats.f.cdf(f_stat, k - 1, dof)

    print(f"\n  Overall Model F-Test:")
    print(f"    • F-Statistic: {f_stat:.6f}")
    print(f"    • P-Value: {f_pvalue:.2e}")
    if f_pvalue < 0.05:
        print(f"    • Result: ✓ Model is statistically significant (p < 0.05)")
    else:
        print(f"    • Result: ✗ Model is NOT statistically significant (p ≥ 0.05)")

    print(f"\n  Top 10 Significant Variables (by absolute T-statistic):")
    print(f"  ─" * 100)

    # Create coefficient summary
    var_names = ['const'] + list(X_orig.columns)
    coef_summary = pd.DataFrame({
        'Variable': var_names,
        'Coefficient': beta,
        'Std Error': std_errors,
        'T-Statistic': t_stats,
        'P-Value': p_values,
        'Significant': ['Yes' if p < 0.05 else 'No' for p in p_values]
    }).sort_values('T-Statistic', key=abs, ascending=False)

    print(coef_summary.head(10).to_string(index=False))

    sig_count = (coef_summary['P-Value'].iloc[1:] < 0.05).sum()  # Exclude constant
    total_vars = len(coef_summary) - 1  # Exclude constant
    print(f"\n  Summary: {sig_count} out of {total_vars} variables are significant (p < 0.05)")

    return {
        'name': 'OLS (Original)',
        'model': ols_model,
        'coef_summary': coef_summary,
        'X': X_orig,
        'y': y,
        'y_pred': y_pred,
        'residuals': residuals,
        'r2': r2,
        'adj_r2': adj_r2,
        'rmse': rmse,
        'mae': mae,
        'mape': mape,
        'cv_scores': cv_scores,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
    }


In [49]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: TRANSFORMATIONS FOR HOMOSCEDASTICITY
# ═══════════════════════════════════════════════════════════════════════════════

def apply_transformations(X, y, y_log):
    """Apply transformations to achieve homoscedasticity."""
    print_section("STEP 6: TRANSFORMATIONS FOR HOMOSCEDASTICITY")

    print(f"\n  Original Target Skewness: {stats.skew(y):.6f}")
    print(f"  Log-transformed Target Skewness: {stats.skew(y_log):.6f}")
    print(f"  ✓ Log Transform applied (target): Skewness improvement confirmed")

    # ── Yeo-Johnson on right-skewed features ──
    print(f"\n  Identifying right-skewed features (|skew| > 1)...")
    skewed_features = []
    for col in X.columns:
        if pd.api.types.is_numeric_dtype(X[col]):
            skew_val = stats.skew(X[col])
            if skew_val > 1:
                skewed_features.append(col)

    print(f"    Found {len(skewed_features)} right-skewed features")

    X_transformed = X.copy()
    if skewed_features:
        pt = PowerTransformer(method='yeo-johnson', standardize=False)
        X_transformed[skewed_features] = pt.fit_transform(X[skewed_features])
        print(f"    ✓ Applied Yeo-Johnson to {len(skewed_features)} features")

    # ── Winsorization (clip outliers at 1st and 99th percentile) ──
    print(f"\n  Applying Winsorization (1st-99th percentile)...")
    X_winsorized = X_transformed.copy()
    for col in X_winsorized.columns:
        if pd.api.types.is_numeric_dtype(X_winsorized[col]):
            try:
                p1 = X_winsorized[col].quantile(0.01)
                p99 = X_winsorized[col].quantile(0.99)
                if pd.notna(p1) and pd.notna(p99):
                    X_winsorized[col] = X_winsorized[col].clip(p1, p99)
            except:
                pass  # Skip if quantile calculation fails
    print(f"    ✓ Clipped outliers for all numeric features")

    # ── Standardization ──
    print(f"\n  Standardizing features (mean=0, std=1)...")
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(
        scaler.fit_transform(X_winsorized),
        columns=X_winsorized.columns,
        index=X_winsorized.index
    )
    print(f"    ✓ Scaled all {len(X_scaled.columns)} features")

    return X_scaled, y_log, scaler


In [57]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 7: OLS ON TRANSFORMED VARIABLES
# ═══════════════════════════════════════════════════════════════════════════════

def run_ols_transformed(X_scaled, y_log, predictors):
    """Run OLS on transformed variables."""
    print_section("STEP 7: OLS REGRESSION (TRANSFORMED VARIABLES)")

    ols_model = LinearRegression()
    ols_model.fit(X_scaled, y_log)

    y_pred = ols_model.predict(X_scaled)
    residuals = y_log - y_pred

    r2 = r2_score(y_log, y_pred)
    rmse = np.sqrt(mean_squared_error(y_log, y_pred))
    mae = mean_absolute_error(y_log, y_pred)
    mape = mean_absolute_percentage_error(y_log, y_pred)
    adj_r2 = 1 - (1 - r2) * (len(y_log) - 1) / (len(y_log) - X_scaled.shape[1] - 1)

    print(f"\n  Model Performance:")
    print(f"    • R² Score: {r2:.6f}")
    print(f"    • Adjusted R²: {adj_r2:.6f}")
    print(f"    • RMSE: {rmse:.6f}")
    print(f"    • MAE: {mae:.6f}")
    print(f"    • MAPE: {mape:.6f}")

    print(f"\n  Residual Analysis:")
    print(f"    • Mean: {residuals.mean():.6f}")
    print(f"    • Std Dev: {residuals.std():.6f}")
    print(f"    • Skewness: {stats.skew(residuals):.6f}")
    print(f"    • Kurtosis: {stats.kurtosis(residuals):.6f}")

    cv_scores = cross_val_score(ols_model, X_scaled, y_log, cv=KFold(n_splits=10, shuffle=True, random_state=42), scoring='r2')
    print(f"\n  10-Fold Cross-Validation:")
    print(f"    • Mean R²: {cv_scores.mean():.6f}")
    print(f"    • Std Dev: {cv_scores.std():.6f}")

    # Statistical Significance Testing (manual implementation)
    print_subsection("Statistical Significance Testing (OLS - Transformed Variables)")

    # Add constant for significance testing
    X_numeric = X_scaled.values.astype(float)
    X_with_const = np.column_stack([np.ones(len(X_numeric)), X_numeric])

    # Calculate coefficients
    beta = np.linalg.lstsq(X_with_const, y_log, rcond=None)[0]
    y_pred_sig = X_with_const @ beta
    residuals_sig = y_log - y_pred_sig

    # Calculate residual sum of squares and standard error
    rss = np.sum(residuals_sig ** 2)
    n = len(y_log)
    k = X_with_const.shape[1]
    dof = n - k
    mse = rss / dof

    # Calculate standard errors
    try:
        var_covar = mse * np.linalg.inv(X_with_const.T @ X_with_const)
        std_errors = np.sqrt(np.diag(var_covar))
    except:
        std_errors = np.ones(k) * np.sqrt(mse)

    # Calculate t-statistics and p-values
    t_stats = beta / (std_errors + 1e-10)
    p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), dof))

    # Calculate F-statistic
    ss_total = np.sum((y_log - y_log.mean()) ** 2)
    ss_res = rss
    ss_reg = ss_total - ss_res
    f_stat = (ss_reg / (k - 1)) / (ss_res / dof)
    f_pvalue = 1 - stats.f.cdf(f_stat, k - 1, dof)

    print(f"\n  Overall Model F-Test:")
    print(f"    • F-Statistic: {f_stat:.6f}")
    print(f"    • P-Value: {f_pvalue:.2e}")
    if f_pvalue < 0.05:
        print(f"    • Result: ✓ Model is statistically significant (p < 0.05)")
    else:
        print(f"    • Result: ✗ Model is NOT statistically significant (p ≥ 0.05)")

    print(f"\n  Top 10 Significant Variables (by absolute T-statistic):")
    print(f"  ─" * 100)

    # Create coefficient summary
    var_names = ['const'] + list(X_scaled.columns)
    coef_summary = pd.DataFrame({
        'Variable': var_names,
        'Coefficient': beta,
        'Std Error': std_errors,
        'T-Statistic': t_stats,
        'P-Value': p_values,
        'Significant': ['Yes' if p < 0.05 else 'No' for p in p_values]
    }).sort_values('T-Statistic', key=abs, ascending=False)

    print(coef_summary.head(10).to_string(index=False))

    sig_count = (coef_summary['P-Value'].iloc[1:] < 0.05).sum()  # Exclude constant
    total_vars = len(coef_summary) - 1  # Exclude constant
    print(f"\n  Summary: {sig_count} out of {total_vars} variables are significant (p < 0.05)")

    return {
        'name': 'OLS (Transformed)',
        'model': ols_model,
        'coef_summary': coef_summary,
        'X': X_scaled,
        'y': y_log,
        'y_pred': y_pred,
        'residuals': residuals,
        'r2': r2,
        'adj_r2': adj_r2,
        'rmse': rmse,
        'mae': mae,
        'mape': mape,
        'cv_scores': cv_scores,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
    }



In [51]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 8: RIDGE REGRESSION
# ═══════════════════════════════════════════════════════════════════════════════

def run_ridge(X_scaled, y_log):
    """Run Ridge Regression with optimal alpha."""
    print_section("STEP 8: RIDGE REGRESSION (TRANSFORMED VARIABLES)")

    # Find optimal alpha via CV
    print(f"\n  Tuning alpha parameter (10-fold CV)...")
    alphas = np.logspace(-4, 4, 50)
    cv_results = []

    for alpha in alphas:
        ridge = Ridge(alpha=alpha)
        scores = cross_val_score(ridge, X_scaled, y_log, cv=KFold(n_splits=10, shuffle=True, random_state=42), scoring='r2')
        cv_results.append(scores.mean())

    best_alpha = alphas[np.argmax(cv_results)]
    print(f"    ✓ Optimal alpha: {best_alpha:.6f}")

    ridge_model = Ridge(alpha=best_alpha)
    ridge_model.fit(X_scaled, y_log)

    y_pred = ridge_model.predict(X_scaled)
    residuals = y_log - y_pred

    r2 = r2_score(y_log, y_pred)
    rmse = np.sqrt(mean_squared_error(y_log, y_pred))
    mae = mean_absolute_error(y_log, y_pred)
    mape = mean_absolute_percentage_error(y_log, y_pred)
    adj_r2 = 1 - (1 - r2) * (len(y_log) - 1) / (len(y_log) - X_scaled.shape[1] - 1)

    print(f"\n  Model Performance:")
    print(f"    • R² Score: {r2:.6f}")
    print(f"    • Adjusted R²: {adj_r2:.6f}")
    print(f"    • RMSE: {rmse:.6f}")
    print(f"    • MAE: {mae:.6f}")
    print(f"    • MAPE: {mape:.6f}")

    cv_scores = cross_val_score(ridge_model, X_scaled, y_log, cv=KFold(n_splits=10, shuffle=True, random_state=42), scoring='r2')
    print(f"\n  10-Fold Cross-Validation:")
    print(f"    • Mean R²: {cv_scores.mean():.6f}")
    print(f"    • Std Dev: {cv_scores.std():.6f}")

    return {
        'name': 'Ridge',
        'model': ridge_model,
        'X': X_scaled,
        'y': y_log,
        'y_pred': y_pred,
        'residuals': residuals,
        'r2': r2,
        'adj_r2': adj_r2,
        'rmse': rmse,
        'mae': mae,
        'mape': mape,
        'cv_scores': cv_scores,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'alpha': best_alpha,
    }


In [52]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 9: PRINCIPAL COMPONENT REGRESSION
# ═══════════════════════════════════════════════════════════════════════════════

def run_pcr(X_scaled, y_log):
    """Run Principal Component Regression."""
    print_section("STEP 9: PRINCIPAL COMPONENT REGRESSION")

    # Determine optimal number of components (Kaiser criterion: EV > 1)
    # Determine optimal number of components (Cumulative Variance ≥ 90%)
    pca = PCA()
    pca.fit(X_scaled)

    cumsum = np.cumsum(pca.explained_variance_ratio_)

    # 🔥 Select number of components to explain ≥ 90% variance
    variance_threshold = 0.90
    n_components = np.argmax(cumsum >= variance_threshold) + 1

    # Safety check (avoid edge cases)
    n_components = min(n_components, X_scaled.shape[1])

    print(f"\n  PCA Analysis:")
    print(f"    • Total features: {X_scaled.shape[1]}")
    print(f"    • Variance threshold: {variance_threshold*100:.0f}%")
    print(f"    • Selected components: {n_components}")
    print(f"    • Variance explained: {cumsum[n_components-1]:.4f}")

    pca_final = PCA(n_components=n_components)
    X_pca = pca_final.fit_transform(X_scaled)

    print(f"    • Reduced feature space: {X_pca.shape}")

    # OLS on PCA components
    pcr_model = LinearRegression()
    pcr_model.fit(X_pca, y_log)

    y_pred = pcr_model.predict(X_pca)
    residuals = y_log - y_pred

    r2 = r2_score(y_log, y_pred)
    rmse = np.sqrt(mean_squared_error(y_log, y_pred))
    mae = mean_absolute_error(y_log, y_pred)
    mape = mean_absolute_percentage_error(y_log, y_pred)
    adj_r2 = 1 - (1 - r2) * (len(y_log) - 1) / (len(y_log) - n_components - 1)

    print(f"\n  Model Performance:")
    print(f"    • R² Score: {r2:.6f}")
    print(f"    • Adjusted R²: {adj_r2:.6f}")
    print(f"    • RMSE: {rmse:.6f}")
    print(f"    • MAE: {mae:.6f}")
    print(f"    • MAPE: {mape:.6f}")

    cv_scores = cross_val_score(pcr_model, X_pca, y_log, cv=KFold(n_splits=10, shuffle=True, random_state=42), scoring='r2')
    print(f"\n  10-Fold Cross-Validation:")
    print(f"    • Mean R²: {cv_scores.mean():.6f}")
    print(f"    • Std Dev: {cv_scores.std():.6f}")

    return {
        'name': 'PCR',
        'model': pcr_model,
        'pca': pca_final,
        'X': X_pca,
        'y': y_log,
        'y_pred': y_pred,
        'residuals': residuals,
        'r2': r2,
        'adj_r2': adj_r2,
        'rmse': rmse,
        'mae': mae,
        'mape': mape,
        'cv_scores': cv_scores,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'n_components': n_components,
    }


In [53]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 10: RANDOM FOREST
# ═══════════════════════════════════════════════════════════════════════════════

def run_random_forest(X_scaled, y_log):
    """Run Random Forest Regression."""
    print_section("STEP 10: RANDOM FOREST REGRESSION")

    rf_model = RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(X_scaled, y_log)

    y_pred = rf_model.predict(X_scaled)
    residuals = y_log - y_pred

    r2 = r2_score(y_log, y_pred)
    rmse = np.sqrt(mean_squared_error(y_log, y_pred))
    mae = mean_absolute_error(y_log, y_pred)
    mape = mean_absolute_percentage_error(y_log, y_pred)
    adj_r2 = 1 - (1 - r2) * (len(y_log) - 1) / (len(y_log) - X_scaled.shape[1] - 1)

    print(f"\n  Model Performance:")
    print(f"    • R² Score: {r2:.6f}")
    print(f"    • Adjusted R²: {adj_r2:.6f}")
    print(f"    • RMSE: {rmse:.6f}")
    print(f"    • MAE: {mae:.6f}")
    print(f"    • MAPE: {mape:.6f}")

    cv_scores = cross_val_score(rf_model, X_scaled, y_log, cv=KFold(n_splits=10, shuffle=True, random_state=42), scoring='r2')
    print(f"\n  10-Fold Cross-Validation:")
    print(f"    • Mean R²: {cv_scores.mean():.6f}")
    print(f"    • Std Dev: {cv_scores.std():.6f}")

    return {
        'name': 'Random Forest',
        'model': rf_model,
        'X': X_scaled,
        'y': y_log,
        'y_pred': y_pred,
        'residuals': residuals,
        'r2': r2,
        'adj_r2': adj_r2,
        'rmse': rmse,
        'mae': mae,
        'mape': mape,
        'cv_scores': cv_scores,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
    }


In [54]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 12: COMPREHENSIVE METRICS TABLE
# ═══════════════════════════════════════════════════════════════════════════════

def create_metrics_table(results):
    """Create comprehensive comparison table."""
    print_section("STEP 12: COMPREHENSIVE METRICS COMPARISON TABLE")

    metrics_data = []

    for result in results:
        metrics_data.append({
            'Model': result['name'],
            'R² Score': f"{result['r2']:.6f}",
            'Adj. R²': f"{result['adj_r2']:.6f}",
            'RMSE': f"{result['rmse']:.6f}",
            'MAE': f"{result['mae']:.6f}",
            'MAPE': f"{result['mape']:.6f}",
            'CV R² Mean': f"{result['cv_mean']:.6f}",
            'CV R² Std': f"{result['cv_std']:.6f}",
            'Residual Skew': f"{stats.skew(result['residuals']):.6f}",
            'Residual Kurt': f"{stats.kurtosis(result['residuals']):.6f}",
        })

    metrics_df = pd.DataFrame(metrics_data)

    print("\n" + metrics_df.to_string(index=False))

    # Save to CSV
    metrics_df.to_csv('/mnt/user-data/outputs/model_metrics_comparison.csv', index=False)
    print(f"\n  ✓ Saved to: model_metrics_comparison.csv")

    return metrics_df



In [68]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 13: VISUALIZATIONS
# ═══════════════════════════════════════════════════════════════════════════════

def create_visualizations(ols_original, ols_transformed, ridge, pcr, rf, X_scaled, X, predictors):
    """Create comprehensive visualization plots."""
    print_section("STEP 13: GENERATING VISUALIZATIONS")

    apply_theme()

    # Setup models list for all plots
    models = [ols_original, ols_transformed, ridge, pcr, rf]

    # ── Plot 0: VIF Analysis ──
    fig, ax = plt.subplots(figsize=(12, 8))

    vif_data = calculate_vif(X_scaled)
    top_vif = vif_data.head(15)

    colors = [ACCENT2 if v > 10 else (ACCENT5 if v > 5 else ACCENT3) for v in top_vif['VIF']]

    bars = ax.barh(range(len(top_vif)), top_vif['VIF'], color=colors, alpha=0.8, edgecolor=TEXT_CLR, linewidth=2)
    ax.set_yticks(range(len(top_vif)))
    ax.set_yticklabels(top_vif['Feature'], fontsize=11)
    ax.axvline(5, color=ACCENT5, linestyle='--', linewidth=2, label='VIF=5 (Moderate)', alpha=0.7)
    ax.axvline(10, color=ACCENT2, linestyle='--', linewidth=2, label='VIF=10 (High)', alpha=0.7)
    ax.set_xlabel('VIF Score', fontsize=12, fontweight='bold')
    ax.set_title('Multicollinearity Check: Top 15 VIF Scores', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    ax.legend(fontsize=10, loc='lower right')
    ax.grid(axis='x', alpha=0.3)

    for i, (idx, row) in enumerate(top_vif.iterrows()):
        ax.text(row['VIF'] + 0.3, i, f"{row['VIF']:.2f}", va='center', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/00_vif_multicollinearity.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print("  ✓ 00_vif_multicollinearity.png")

    # ── Plot 1a: OLS Coefficient Significance (Original) ──
    fig, ax = plt.subplots(figsize=(12, 8))

    if 'coef_summary' in ols_original:
        coef_data = ols_original['coef_summary'].iloc[1:11]  # Top 10 (exclude const)
        colors_sig = [ACCENT3 if p < 0.05 else ACCENT2 for p in coef_data['P-Value']]

        ax.barh(range(len(coef_data)), coef_data['Coefficient'], color=colors_sig, alpha=0.8, edgecolor=TEXT_CLR, linewidth=2)
        ax.set_yticks(range(len(coef_data)))
        ax.set_yticklabels(coef_data['Variable'], fontsize=11)
        ax.set_xlabel('Coefficient Value', fontsize=12, fontweight='bold')
        ax.set_title('OLS (Original): Top 10 Variable Coefficients\n(Green = Significant at p<0.05, Red = Not Significant)',
                    fontsize=13, fontweight='bold')
        ax.invert_yaxis()
        ax.grid(axis='x', alpha=0.3)

        # Add significance stars
        for i, (idx, row) in enumerate(coef_data.iterrows()):
            sig_marker = '***' if row['P-Value'] < 0.001 else ('**' if row['P-Value'] < 0.01 else ('*' if row['P-Value'] < 0.05 else 'ns'))
            ax.text(row['Coefficient'], i, f"  {sig_marker}", va='center', fontsize=10, fontweight='bold')

        plt.tight_layout()
        plt.savefig('/mnt/user-data/outputs/01_ols_original_coefficients.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
        plt.close()
        print("  ✓ 01_ols_original_coefficients.png")

    # ── Plot 1b: OLS Coefficient Significance (Transformed) ──
    fig, ax = plt.subplots(figsize=(12, 8))

    if 'coef_summary' in ols_transformed:
        coef_data = ols_transformed['coef_summary'].iloc[1:11]  # Top 10 (exclude const)
        colors_sig = [ACCENT3 if p < 0.05 else ACCENT2 for p in coef_data['P-Value']]

        ax.barh(range(len(coef_data)), coef_data['Coefficient'], color=colors_sig, alpha=0.8, edgecolor=TEXT_CLR, linewidth=2)
        ax.set_yticks(range(len(coef_data)))
        ax.set_yticklabels(coef_data['Variable'], fontsize=11)
        ax.set_xlabel('Coefficient Value', fontsize=12, fontweight='bold')
        ax.set_title('OLS (Transformed): Top 10 Variable Coefficients\n(Green = Significant at p<0.05, Red = Not Significant)',
                    fontsize=13, fontweight='bold')
        ax.invert_yaxis()
        ax.grid(axis='x', alpha=0.3)

        # Add significance stars
        for i, (idx, row) in enumerate(coef_data.iterrows()):
            sig_marker = '***' if row['P-Value'] < 0.001 else ('**' if row['P-Value'] < 0.01 else ('*' if row['P-Value'] < 0.05 else 'ns'))
            ax.text(row['Coefficient'], i, f"  {sig_marker}", va='center', fontsize=10, fontweight='bold')

        plt.tight_layout()
        plt.savefig('/mnt/user-data/outputs/01b_ols_transformed_coefficients.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
        plt.close()
        print("  ✓ 01b_ols_transformed_coefficients.png")

    # ── Plot 2: R² Comparison ──
    fig, ax = plt.subplots(figsize=(12, 6))
    r2_scores = [m['r2'] for m in models]
    colors = [ACCENT1, ACCENT2, ACCENT3, ACCENT4, ACCENT5]

    bars = ax.bar([m['name'] for m in models], r2_scores, color=colors, alpha=0.8, edgecolor=TEXT_CLR, linewidth=2)
    ax.set_ylabel('R² Score', fontsize=12, fontweight='bold')
    ax.set_title('Model Performance Comparison: R² Scores', fontsize=14, fontweight='bold')
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)

    for bar, score in zip(bars, r2_scores):
        ax.text(bar.get_x() + bar.get_width()/2, score + 0.02, f'{score:.4f}',
                ha='center', va='bottom', fontsize=11, fontweight='bold', color=TEXT_CLR)

    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/02_r2_comparison.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print("  ✓ 02_r2_comparison.png")

    # ── Plot 3: Metrics Comparison (RMSE, MAE, MAPE) ──
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    rmse_vals = [m['rmse'] for m in models]
    mae_vals = [m['mae'] for m in models]
    mape_vals = [m['mape'] for m in models]
    model_names = [m['name'] for m in models]

    for ax, vals, title in zip(axes, [rmse_vals, mae_vals, mape_vals], ['RMSE', 'MAE', 'MAPE']):
        bars = ax.bar(model_names, vals, color=colors, alpha=0.8, edgecolor=TEXT_CLR, linewidth=2)
        ax.set_ylabel(title, fontsize=12, fontweight='bold')
        ax.set_title(f'{title} Comparison', fontsize=12, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)

        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, val + max(vals)*0.02, f'{val:.4f}',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.suptitle('Error Metrics Comparison', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/03_error_metrics_comparison.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print("  ✓ 03_error_metrics_comparison.png")

    # ── Plot 4: Cross-Validation Results ──
    fig, ax = plt.subplots(figsize=(12, 6))

    x_pos = np.arange(len(models))
    cv_means = [m['cv_mean'] for m in models]
    cv_stds = [m['cv_std'] for m in models]

    bars = ax.bar(x_pos, cv_means, yerr=cv_stds, color=colors, alpha=0.8, capsize=10,
                  edgecolor=TEXT_CLR, linewidth=2, error_kw={'elinewidth': 2, 'capthick': 2})

    ax.set_ylabel('Mean CV R² Score', fontsize=12, fontweight='bold')
    ax.set_title('10-Fold Cross-Validation Results', fontsize=14, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(model_names)
    ax.grid(axis='y', alpha=0.3)

    for i, (bar, mean, std) in enumerate(zip(bars, cv_means, cv_stds)):
        ax.text(bar.get_x() + bar.get_width()/2, mean + std + 0.01, f'{mean:.4f}\n±{std:.4f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/04_crossvalidation_results.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print("  ✓ 04_crossvalidation_results.png")

    # ── Plot 5: Predicted vs Actual (OLS Original) ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # OLS Original
    ax = axes[0]
    ax.scatter(ols_original['y'], ols_original['y_pred'], alpha=0.5, s=20, color=ACCENT1, edgecolors='none')
    lims = [min(ols_original['y'].min(), ols_original['y_pred'].min()),
            max(ols_original['y'].max(), ols_original['y_pred'].max())]
    ax.plot(lims, lims, 'r--', lw=2, label='Perfect fit')
    ax.set_xlabel('Actual Lifetime Value', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted Lifetime Value', fontsize=11, fontweight='bold')
    ax.set_title(f'OLS (Original) - R²={ols_original["r2"]:.4f}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

    # Random Forest
    ax = axes[1]
    ax.scatter(rf['y'], rf['y_pred'], alpha=0.5, s=20, color=ACCENT5, edgecolors='none')
    lims_rf = [min(rf['y'].min(), rf['y_pred'].min()),
               max(rf['y'].max(), rf['y_pred'].max())]
    ax.plot(lims_rf, lims_rf, 'r--', lw=2, label='Perfect fit')
    ax.set_xlabel('Log(Lifetime Value)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted Log(LTV)', fontsize=11, fontweight='bold')
    ax.set_title(f'Random Forest - R²={rf["r2"]:.4f}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

    plt.suptitle('Predicted vs Actual Values', fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/05_predicted_vs_actual.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print("  ✓ 05_predicted_vs_actual.png")

    # ── Plot 6: Residual Distributions ──
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()

    for i, model in enumerate(models):
        ax = axes[i]
        ax.hist(model['residuals'], bins=50, color=colors[i], alpha=0.7, edgecolor=TEXT_CLR)
        ax.axvline(model['residuals'].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
        ax.set_xlabel('Residuals', fontsize=11, fontweight='bold')
        ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
        ax.set_title(f"{model['name']} Residuals\n(Skew={stats.skew(model['residuals']):.4f})",
                    fontsize=11, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

    # Hide last subplot
    axes[-1].axis('off')

    plt.suptitle('Residual Distributions', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/06_residual_distributions.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print("  ✓ 06_residual_distributions.png")

    # ── Plot 7: Q-Q Plots ──
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()

    for i, model in enumerate(models):
        ax = axes[i]
        resid_std = (model['residuals'] - model['residuals'].mean()) / model['residuals'].std()
        stats.probplot(resid_std, dist="norm", plot=ax)
        ax.set_title(f"{model['name']} Q-Q Plot", fontsize=11, fontweight='bold')
        ax.grid(alpha=0.3)

    axes[-1].axis('off')

    plt.suptitle('Normality of Residuals (Q-Q Plots)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/07_qq_plots.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print("  ✓ 07_qq_plots.png")

    # ── Plot 8: Feature Importance (Random Forest) ──
    fig, ax = plt.subplots(figsize=(12, 8))

    importances = rf['model'].feature_importances_
    indices = np.argsort(importances)[::-1][:15]

    ax.barh(range(len(indices)), importances[indices], color=ACCENT5, alpha=0.8, edgecolor=TEXT_CLR, linewidth=2)
    ax.set_yticks(range(len(indices)))
    ax.set_yticklabels([X_scaled.columns[i] for i in indices], fontsize=11)
    ax.set_xlabel('Importance Score', fontsize=12, fontweight='bold')
    ax.set_title('Random Forest: Top 15 Feature Importances', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)

    for i, v in enumerate(importances[indices]):
        ax.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/08_feature_importance.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print("  ✓ 08_feature_importance.png")

    # ── Plot 9: Adjusted R² Comparison ──
    fig, ax = plt.subplots(figsize=(12, 6))

    adj_r2_scores = [m['adj_r2'] for m in models]

    bars = ax.bar(model_names, adj_r2_scores, color=colors, alpha=0.8, edgecolor=TEXT_CLR, linewidth=2)
    ax.set_ylabel('Adjusted R² Score', fontsize=12, fontweight='bold')
    ax.set_title('Adjusted R² Comparison (Accounting for Model Complexity)', fontsize=14, fontweight='bold')
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)

    for bar, score in zip(bars, adj_r2_scores):
        ax.text(bar.get_x() + bar.get_width()/2, score + 0.02, f'{score:.4f}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/09_adj_r2_comparison.png', dpi=300, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print("  ✓ 09_adj_r2_comparison.png")

    # ── Plot 10: Residuals vs Fitted (RAW vs LOG) ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    import statsmodels.api as sm

    X_const = sm.add_constant(X_scaled)

    # RAW
    model_raw = sm.OLS(ols_original['y'], X_const).fit()
    axes[0].scatter(model_raw.fittedvalues, model_raw.resid, alpha=0.5)
    axes[0].axhline(0, linestyle='--')
    axes[0].set_title("Residuals vs Fitted (RAW LTV)")

    # LOG
    model_log = sm.OLS(ols_transformed['y'], X_const).fit()
    axes[1].scatter(model_log.fittedvalues, model_log.resid, alpha=0.5)
    axes[1].axhline(0, linestyle='--')
    axes[1].set_title("Residuals vs Fitted (LOG LTV)")

    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/10_residuals_comparison.png', dpi=300)
    plt.close()

    print("  ✓ 10_residuals_comparison.png")

    print(f"\n  Total visualizations created: 12")

In [69]:
# ═══════════════════════════════════════════════════════════════════════════════
# MAIN EXECUTION
# ═══════════════════════════════════════════════════════════════════════════════

def main():
    """Execute complete analysis pipeline."""

    print("\n" + "=" * 100)
    print("  COMPREHENSIVE LTV PREDICTION ANALYSIS - START")
    print(f"  Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 100)

    # Step 1
    df = load_and_prepare_data()

    # Step 2
    df = feature_engineering(df)

    # Step 3
    descriptive_statistics(df)

    # Step 4
    X, y, y_log, predictors = prepare_model_data(df)

    # Step 4A: MULTICOLLINEARITY CHECK (VIF)
    print_section("STEP 4A: MULTICOLLINEARITY CHECK (VARIANCE INFLATION FACTOR)")

    vif_analysis = calculate_vif(X)
    print(f"\n  VIF Analysis (Multicollinearity Assessment):")
    print(f"  ─" * 100)
    print(f"  VIF Interpretation:")
    print(f"    • VIF < 5: Low multicollinearity ✓")
    print(f"    • VIF 5-10: Moderate multicollinearity ⚠")
    print(f"    • VIF > 10: High multicollinearity ✗")
    print(f"\n  Top 15 Features by VIF Score:")
    print(vif_analysis.head(15).to_string(index=False))

    high_vif_count = (vif_analysis['VIF'] > 10).sum()
    moderate_vif_count = ((vif_analysis['VIF'] >= 5) & (vif_analysis['VIF'] <= 10)).sum()

    print(f"\n  Summary:")
    print(f"    • Features with VIF > 10: {high_vif_count}")
    print(f"    • Features with VIF 5-10: {moderate_vif_count}")
    print(f"    • Features with VIF < 5: {len(vif_analysis) - high_vif_count - moderate_vif_count}")

    if high_vif_count > 0:
        print(f"    ⚠ Warning: {high_vif_count} features show high multicollinearity")
        print(f"      Ridge regression may help regularize these relationships")
    else:
        print(f"    ✓ No severe multicollinearity detected")

    # Step 5
    ols_original = run_ols_original(X, y, predictors)

    # Step 6
    X_scaled, y_transformed, scaler = apply_transformations(X, y, y_log)

    # Step 7
    ols_transformed = run_ols_transformed(X_scaled, y_transformed, predictors)

    # Step 8
    ridge = run_ridge(X_scaled, y_transformed)

    # Step 9
    pcr = run_pcr(X_scaled, y_transformed)

    # Step 10
    rf = run_random_forest(X_scaled, y_transformed)

    # Step 11
    results = [ols_original, ols_transformed, ridge, pcr, rf]
    # stat_tests = perform_statistical_tests(results)

    # Step 12
    metrics_df = create_metrics_table(results)

    # Step 13
    create_visualizations(ols_original, ols_transformed, ridge, pcr, rf, X_scaled, X, predictors)

    # Summary
    print_section("ANALYSIS COMPLETE")
    print(f"\n  ✓ All models trained and evaluated")
    print(f"  ✓ Multicollinearity analysis (VIF) completed")
    print(f"  ✓ Statistical significance tests (F-test, T-tests) performed for OLS models")
    print(f"  ✓ 10-fold cross-validation completed for all models")
    print(f"  ✓ Comprehensive metrics table created")
    print(f"  ✓ 11 professional visualizations generated")
    print(f"\n  Output files saved to: /mnt/user-data/outputs/")
    print(f"    • model_metrics_comparison.csv")
    print(f"    • 00_vif_multicollinearity.png")
    print(f"    • 01_ols_original_coefficients.png")
    print(f"    • 01b_ols_transformed_coefficients.png")
    print(f"    • 02_r2_comparison.png through 09_adj_r2_comparison.png")
    print(f"\n  Best performing model: {results[np.argmax([r['r2'] for r in results])]['name']}")
    print(f"    R² Score: {max([r['r2'] for r in results]):.6f}")

    print("\n" + "=" * 100)
    print(f"  Analysis completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 100 + "\n")

if __name__ == "__main__":
    main()


  COMPREHENSIVE LTV PREDICTION ANALYSIS - START
  Timestamp: 2026-04-14 11:35:45

════════════════════════════════════════════════════════════════════════════════════════════════════
  STEP 1: DATA LOADING & PREPARATION
════════════════════════════════════════════════════════════════════════════════════════════════════

  Initial Dataset:
    • Shape: (50000, 25)
    • Columns: 25
    • Missing values: 49081

  Missing value handling:
    • Age: 2495 values → median imputation
    • Session_Duration_Avg: 3399 values → median imputation
    • Pages_Per_Session: 3000 values → median imputation
    • Wishlist_Items: 4000 values → median imputation
    • Days_Since_Last_Purchase: 3000 values → median imputation
    • Discount_Usage_Rate: 3500 values → median imputation
    • Returns_Rate: 4491 values → median imputation
    • Email_Open_Rate: 2528 values → median imputation
    • Customer_Service_Calls: 168 values → median imputation
    • Product_Reviews_Written: 3500 values → median imp